# Task 3: Correlation between News Sentiment and Stock Returns

## Objective
This analysis explores the relationship between financial news sentiment and stock market returns by:
- Aligning news data with trading days
- Computing sentiment scores using VADER
- Calculating daily stock returns
- Measuring correlation between sentiment and returns
import pandas as pd
import matplotlib.pyplot as plt
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

stock_df = pd.read_csv("../data/raw/stock_data.csv")
news_df = pd.read_csv("../data/raw/news_data.csv")
stock_df['Date'] = pd.to_datetime(stock_df['Date'])
news_df['date'] = pd.to_datetime(news_df['date'])

stock_df['trading_date'] = stock_df['Date'].dt.date
news_df['news_date'] = news_df['date'].dt.date

stock_df['trading_date'] = pd.to_datetime(stock_df['trading_date'])
news_df['news_date'] = pd.to_datetime(news_df['news_date'])

trading_days = stock_df[['trading_date']].drop_duplicates().sort_values('trading_date')

aligned_news = pd.merge_asof(
    news_df.sort_values('news_date'),
    trading_days.sort_values('trading_date'),
    left_on='news_date',
    right_on='trading_date',
    direction='forward'
)

aligned_news.rename(columns={'trading_date': 'Date'}, inplace=True)
analyzer = SentimentIntensityAnalyzer()

aligned_news['sentiment'] = aligned_news['headline'].astype(str).apply(
    lambda x: analyzer.polarity_scores(x)['compound']
)
stock_df['daily_return'] = stock_df['Close'].pct_change() * 100
daily_sentiment = aligned_news.groupby('Date')['sentiment'].mean().reset_index()

returns_df = stock_df[['Date', 'daily_return']]

merged_df = pd.merge(daily_sentiment, returns_df, on='Date', how='inner')
correlation = merged_df['sentiment'].corr(merged_df['daily_return'])

print("Pearson Correlation Coefficient:")
print(correlation)
plt.figure(figsize=(8,5))

plt.scatter(merged_df['sentiment'], merged_df['daily_return'])

plt.title("Sentiment vs Stock Returns")
plt.xlabel("Average Daily Sentiment")
plt.ylabel("Daily Return (%)")

plt.text(
    0.05, 0.95,
    f"Correlation = {correlation:.4f}",
    transform=plt.gca().transAxes
)

plt.show()
## Interpretation of Results

The correlation between news sentiment and stock returns indicates the strength and direction of their relationship.

If the correlation is positive, it suggests that positive news sentiment is associated with higher stock returns. A negative correlation suggests that negative sentiment aligns with lower returns. However, if the correlation is close to zero, it indicates a weak or no linear relationship.

This result may be affected by market noise, lag effects, and external macroeconomic factors that are not captured in the sentiment analysis.
def classify(score):
    if score > 0.05:
        return "Positive"
    elif score < -0.05:
        return "Negative"
    else:
        return "Neutral"

merged_df['category'] = merged_df['sentiment'].apply(classify)

category_returns = merged_df.groupby('category')['daily_return'].mean()

category_returns.plot(kind='bar')

plt.title("Average Return by Sentiment Category")
plt.ylabel("Average Return (%)")
plt.xlabel("Sentiment Category")
plt.xticks(rotation=0)

plt.show()
